# Web后端开发 05：数据库基础（SQLite）

上一课的学习任务保存在Python列表中：程序一退出，数据就消失。本课把“业务数据放在哪里”继续向下拆一层，学习如何把数据可靠地保存到SQLite数据库文件。

> 这是一节从零开始的数据库课。Notebook会先解释数据库术语，再写SQL和Python代码，不默认你已经学过数据库。

## 1. 本课目标与范围

完成本课后，你应该能够：

1. 用自己的话解释数据库、表、行、列、主键和约束；
2. 区分SQLite数据库文件、Python连接对象和SQL语句；
3. 使用Python标准库`sqlite3`连接数据库并创建表；
4. 使用`INSERT`、`SELECT`、`UPDATE`和`DELETE`完成新增、查询、修改和删除（合称CRUD）；
5. 使用`?`占位符安全绑定Python数据，避免拼接SQL；
6. 理解`commit()`、`rollback()`和`close()`分别解决什么问题；
7. 使用`sqlite3.Row`把查询结果转换为清晰的字典；
8. 编写一个可测试、可持久化的repository（专门访问数据库的一层代码）。

本课暂不学习把表映射成Python对象的ORM、异步访问数据库、多表关系、查询性能优化和表结构版本迁移。这些进阶内容需要建立在本课基础之上。

## 2. 运行本课前：环境和依赖

SQLite已经随Python的`sqlite3`标准库提供，不需要单独下载MySQL或启动数据库服务器。本课的自动测试需要第三方库pytest。

请在PowerShell中进入本课目录并执行：

```powershell
cd "D:\桌面\学点东西\codex—help\02_Web后端开发\05_数据库基础"
python -m pip install -r requirements.txt
python -c "import sqlite3; print(sqlite3.sqlite_version)"
```

- `python`：当前环境的Python解释器；
- `-m pip`：让当前Python运行自己的pip模块；
- `install`：安装依赖；
- `-r requirements.txt`：从依赖文件读取要安装的包；
- `python -c "..."`：直接执行引号中的短代码；这里打印实际SQLite运行库版本。

也可以运行下一格的`%pip`。它会向当前Notebook内核安装pytest；安装后若导入仍失败，请重启内核。

In [ ]:
# 如果已在PowerShell安装，可以跳过这一格。
%pip install -r "D:/桌面/学点东西/codex—help/02_Web后端开发/05_数据库基础/requirements.txt"

`requirements.txt`中只有`pytest`，用于运行课末作业测试。`sqlite3`是标准库，不能也不需要用pip安装。

下面首次导入：

- `sqlite3`：Python标准库模块，提供连接SQLite和执行SQL的接口；
- `pytest`：第三方测试框架，这里只检查安装版本。

`sqlite3.sqlite_version`是字符串，表示Python当前连接的SQLite运行库版本。不要使用已弃用的`sqlite3.version`。

In [ ]:
import sqlite3
import pytest

print("SQLite运行库：", sqlite3.sqlite_version)
print("pytest：", pytest.__version__)

## 3. 为什么列表不够用：什么叫持久化

上一课中的代码大致是：

```python
_tasks = []
_tasks.append(task)
```

列表存在于当前Python进程的内存中。程序结束后，操作系统会回收这块内存，下一次启动又是空列表。

**持久化**表示：程序停止后数据仍然存在，下一次启动可以重新读取。SQLite会把表结构和数据写入一个磁盘文件，例如`notes.sqlite3`。

```text
Python列表：程序运行 → 内存中有数据 → 程序退出 → 数据消失
SQLite：   程序运行 → 写入数据库文件 → 程序退出 → 文件仍存在
```

## 4. 关系数据库的基础词汇

可以把一张数据库表理解成一张规则严格的电子表格：

| id | title | year | watched |
|---:|---|---:|---:|
| 1 | 星际穿越 | 2014 | 1 |
| 2 | 流浪地球 | 2019 | 0 |

- **数据库（database）**：保存一组相关表和数据的整体；SQLite通常对应一个文件；
- **表（table）**：保存同一类对象，例如`movies`表保存电影；
- **列（column）**：一类字段，例如`title`列保存标题；
- **行（row）**：一个完整记录，例如一行代表一部电影；
- **主键（primary key）**：能唯一识别每一行的列；这里是`id`；
- **表结构（schema）**：表有哪些列、类型和约束的规则；不是Pydantic的`schemas.py`文件；
- **约束（constraint）**：数据库拒绝不符合规则的数据，例如`NOT NULL`不允许空值。

`id`、`title`、`year`和`watched`都不是Python关键字，是开发者根据业务定义的数据库列名。

### 课堂小练习1：匹配数据库术语

**目标：**区分表、行、列、主键。

请补全字典`database_terms`。可用答案是`"表"`、`"行"`、`"列"`、`"主键"`，每个只用一次。正确时运行紧接着的检测格会显示✅。

In [ ]:
database_terms = {
    "同一类数据的集合": "请填写",
    "一条完整记录": "请填写",
    "记录中的一个字段": "请填写",
    "唯一识别每条记录": "请填写",
}

In [ ]:
try:
    _check_expected = {
        "同一类数据的集合": "表",
        "一条完整记录": "行",
        "记录中的一个字段": "列",
        "唯一识别每条记录": "主键",
    }
    if "database_terms" not in globals():
        print("❌ 请先运行上方答题单元格。")
    elif not isinstance(database_terms, dict):
        print("❌ database_terms必须是字典。")
    else:
        _check_wrong = [key for key, value in _check_expected.items() if database_terms.get(key) != value]
        if _check_wrong:
            print("❌ 这些描述的术语仍不正确：", _check_wrong)
        else:
            print("✅ 小练习1通过：基础术语匹配正确。")
except Exception as _check_error:
    print("❌ 检测时发现问题：", _check_error)

<details><summary>完成后查看参考答案</summary>

同一类数据的集合是表；一条完整记录是行；记录中的一个字段是列；唯一识别每条记录的是主键。

</details>

## 5. SQLite、SQL和`sqlite3`分别是什么

三个名字容易混淆：

- **SQLite**：真正负责保存和查询数据的轻量数据库引擎；它直接使用文件，不需要单独启动服务器进程；
- **SQL**：Structured Query Language，向关系数据库表达“建表、插入、查询、修改、删除”的语言；
- **`sqlite3`**：Python标准库模块，负责把Python代码和SQLite连接起来。

```text
Python代码
   ↓ 调用sqlite3
sqlite3连接对象
   ↓ 发送SQL
SQLite数据库引擎
   ↓ 读写
.sqlite3数据库文件
```

SQLite适合学习、测试和中小型单机应用。MySQL、PostgreSQL通常是独立数据库服务器，但“表、主键、SQL、事务”等核心概念可以迁移。

### SQLite数据类型与Python类型

本课使用的映射：

| SQLite类型 | 常见Python类型 | 示例 |
|---|---|---|
| `NULL` | `None` | 未提供的评分 |
| `INTEGER` | `int` | 年份、0或1 |
| `REAL` | `float` | 评分8.7 |
| `TEXT` | `str` | 电影标题 |
| `BLOB` | `bytes` | 二进制文件内容 |

SQLite采用灵活类型，但声明类型和约束仍然很重要。布尔值通常保存成`INTEGER`：0代表`False`，1代表`True`；读取后可用`bool(value)`转换。

## 6. 打开数据库连接

首次使用的函数：

```python
sqlite3.connect(database, *, autocommit=False)
```

- 来自标准库`sqlite3`；
- `database`是第一个位置参数，可以是文件路径，也可以是特殊字符串`":memory:"`；
- `":memory:"`表示只在内存中创建临时数据库，连接关闭后消失，适合课堂实验；
- `autocommit=False`是关键字参数，表示写操作需要明确`commit()`或`rollback()`；本课使用这种可预测的事务方式；
- 本课环境是Python 3.13；`autocommit`参数要求Python 3.12及以上。如果你的环境更旧，请先升级Python，不要直接删掉这个参数改变事务行为；
- 返回`sqlite3.Connection`连接对象；
- 连接对象代表Python与数据库之间的一次会话，不是数据库文件本身。

文件不存在时，SQLite通常会在连接时创建它；只打开连接并不会自动创建业务表。

In [ ]:
# 创建只存在于当前内存中的课堂数据库。
demo_connection = sqlite3.connect(":memory:", autocommit=False)

# sqlite3.Row是sqlite3提供的行类型工厂。
# 设置后，之后产生的查询行既能按下标读取，也能按列名读取。
demo_connection.row_factory = sqlite3.Row

print(type(demo_connection))

### Connection和Cursor的区别

- `Connection`代表与数据库的连接，负责事务的`commit()`、`rollback()`和最终`close()`；
- `Cursor`代表一次SQL执行和结果读取位置，可以调用`fetchone()`、`fetchall()`，也提供`lastrowid`和`rowcount`。

本课使用连接对象的快捷方法：

```python
cursor = connection.execute(sql, parameters)
```

- `execute()`是`Connection`的方法；
- `sql`是第一个位置参数，类型是字符串；
- `parameters`是可选的第二个位置参数，通常是元组；
- 返回`Cursor`对象；Python会在内部创建游标，所以用户代码里没有单独的`connection.cursor()`行。

## 7. 使用`CREATE TABLE`定义表结构

下面是本课电影表示例：

```sql
CREATE TABLE IF NOT EXISTS movies (
    id INTEGER PRIMARY KEY,
    title TEXT NOT NULL CHECK(length(title) BETWEEN 1 AND 100),
    year INTEGER NOT NULL CHECK(year >= 1888),
    rating REAL,
    watched INTEGER NOT NULL DEFAULT 0 CHECK(watched IN (0, 1))
)
```

逐项解释：

- `CREATE TABLE`：创建表；
- `IF NOT EXISTS`：表已存在时不重复创建，也不会清空原数据；
- `id INTEGER PRIMARY KEY`：唯一整数主键；插入时省略，SQLite生成；
- `NOT NULL`：不能保存SQL的`NULL`，对应Python的`None`；
- `DEFAULT 0`：插入时省略`watched`，数据库填0；主动传1则保存1；主动传`None`会违反`NOT NULL`；
- `CHECK(...)`：写入时检查表达式，结果不符合就拒绝；
- `rating REAL`没有`NOT NULL`，因此可以省略或保存`None`。

SQL关键字通常大写便于阅读，但SQLite不要求必须大写。

In [ ]:
create_movies_sql = """
CREATE TABLE IF NOT EXISTS movies (
    id INTEGER PRIMARY KEY,
    title TEXT NOT NULL CHECK(length(title) BETWEEN 1 AND 100),
    year INTEGER NOT NULL CHECK(year >= 1888),
    rating REAL,
    watched INTEGER NOT NULL DEFAULT 0 CHECK(watched IN (0, 1))
)
"""

create_cursor = demo_connection.execute(create_movies_sql)
demo_connection.commit()
print("execute返回：", type(create_cursor))

columns = demo_connection.execute("PRAGMA table_info(movies)").fetchall()
print("列名：", [column["name"] for column in columns])

`PRAGMA table_info(movies)`是SQLite提供的表结构查询命令，用于观察列，不属于通用SQL。这里`fetchall()`没有参数，返回所有结果行组成的列表。

错误示例：再次执行不带`IF NOT EXISTS`的`CREATE TABLE movies (...)`会因为同名表已经存在而产生`sqlite3.OperationalError`。`IF NOT EXISTS`只避免重复建表，不会检查旧表结构是否和新SQL完全相同。

### 课堂小练习2：编写建表SQL

**目标：**定义列、主键和约束。

请把SQL字符串赋给`create_courses_sql`，要求创建`courses`表：

- `id`：`INTEGER PRIMARY KEY`；
- `name`：`TEXT NOT NULL`；
- `hours`：`INTEGER NOT NULL`且必须严格大于0；
- 重复执行时不报错。

自动检测会在独立内存数据库中真正执行你的SQL并检查表结构。

In [ ]:
create_courses_sql = """
-- 在这里编写CREATE TABLE语句
"""

In [ ]:
_check_connection = None
try:
    if "create_courses_sql" not in globals():
        print("❌ 请先运行上方答题单元格。")
    elif not isinstance(create_courses_sql, str) or not create_courses_sql.strip():
        print("❌ create_courses_sql必须是非空字符串。")
    else:
        _check_connection = sqlite3.connect(":memory:", autocommit=False)
        _check_connection.execute(create_courses_sql)
        _check_columns = _check_connection.execute("PRAGMA table_info(courses)").fetchall()
        _check_names = [column[1] for column in _check_columns]
        _check_sql_row = _check_connection.execute(
            "SELECT sql FROM sqlite_master WHERE type = ? AND name = ?",
            ("table", "courses"),
        ).fetchone()
        _check_sql = _check_sql_row[0].upper() if _check_sql_row else ""
        if _check_names != ["id", "name", "hours"]:
            print("❌ courses表的列名或顺序不正确，实际是：", _check_names)
        elif _check_columns[0][2].upper() != "INTEGER" or _check_columns[0][5] != 1:
            print("❌ id必须是INTEGER PRIMARY KEY。")
        elif _check_columns[1][2].upper() != "TEXT" or _check_columns[1][3] != 1:
            print("❌ name必须是TEXT NOT NULL。")
        elif _check_columns[2][2].upper() != "INTEGER" or _check_columns[2][3] != 1:
            print("❌ hours必须是INTEGER NOT NULL。")
        elif "CHECK" not in _check_sql or "HOURS" not in _check_sql:
            print("❌ 还没有为hours设置大于0的CHECK约束。")
        elif "IF NOT EXISTS" not in create_courses_sql.upper():
            print("❌ 题目要求重复执行不报错，请使用IF NOT EXISTS。")
        else:
            try:
                _check_connection.execute("INSERT INTO courses(name, hours) VALUES (?, ?)", ("错误课", 0))
                print("❌ hours=0仍能插入，CHECK约束还不正确。")
            except sqlite3.IntegrityError:
                print("✅ 小练习2通过：表结构和约束正确。")
except sqlite3.Error as _check_error:
    print("❌ SQL执行失败：", _check_error)
except Exception as _check_error:
    print("❌ 检测时发现问题：", _check_error)
finally:
    if _check_connection is not None:
        _check_connection.close()

<details><summary>完成后查看参考答案</summary>

需要使用`CREATE TABLE IF NOT EXISTS courses (...)`，三列依次声明为`id INTEGER PRIMARY KEY`、`name TEXT NOT NULL`、`hours INTEGER NOT NULL CHECK(hours > 0)`。

</details>

## 8. CRUD：数据库最常见的四类操作

CRUD是四个英文动作的首字母：

| 动作 | SQL | 本课含义 |
|---|---|---|
| Create | `INSERT` | 新增一行 |
| Read | `SELECT` | 查询行 |
| Update | `UPDATE` | 修改已有行 |
| Delete | `DELETE` | 删除行 |

注意：CRUD中的Create通常指创建一条业务数据，对应`INSERT`；`CREATE TABLE`则是创建表结构，不是同一层面的“创建”。

## 9. `INSERT`与参数占位符

安全插入一部电影：

```python
cursor = connection.execute(
    "INSERT INTO movies(title, year, rating) VALUES (?, ?, ?)",
    ("星际穿越", 2014, 9.4),
)
connection.commit()
```

- SQL中的每个`?`是一个占位符，不是字符串里的问号数据；
- 第二个位置参数是参数元组，元素顺序必须与占位符顺序一致；
- 三个`?`必须对应三个值；数量不匹配会产生`sqlite3.ProgrammingError`；
- `execute()`负责把SQL结构和值分开交给SQLite；
- 返回的Cursor对象的`lastrowid`是本次插入生成的整数主键；
- 写入后调用`commit()`，把当前事务的修改确认保存；它没有参数，返回`None`。

单元素元组必须写成`(value,)`，末尾逗号不能省略；`(value)`只是普通括号表达式。

In [ ]:
first_insert = demo_connection.execute(
    "INSERT INTO movies(title, year, rating) VALUES (?, ?, ?)",
    ("星际穿越", 2014, 9.4),
)
second_insert = demo_connection.execute(
    "INSERT INTO movies(title, year, rating) VALUES (?, ?, ?)",
    ("流浪地球", 2019, None),
)
third_insert = demo_connection.execute(
    "INSERT INTO movies(title, year, rating, watched) VALUES (?, ?, ?, ?)",
    ("机器人总动员", 2008, 9.3, 1),
)
demo_connection.commit()

print("第一条id：", first_insert.lastrowid)
print("第二条id：", second_insert.lastrowid)
print("第三条id：", third_insert.lastrowid)

第二条主动传入`None`，数据库保存为`NULL`，因为`rating`允许空值。`watched`在前两条中被省略，数据库使用`DEFAULT 0`；第三条显式传入1。

### 为什么不能用f字符串拼接SQL

不安全写法：

```python
sql = f"INSERT INTO movies(title) VALUES ('{user_title}')"
```

用户输入可能包含引号和SQL片段，使数据改变SQL结构，这叫SQL注入。正确写法始终是：

```python
sql = "INSERT INTO movies(title) VALUES (?)"
parameters = (user_title,)
```

占位符不是简单替换字符串；数据库驱动会把参数作为数据绑定，即使内容是`"'); DROP TABLE movies; --"`也只会作为普通文本保存。

### 课堂小练习3：准备参数化INSERT

**目标：**把SQL结构和Python数据分开。

请设置：

- `insert_language_sql`：向`languages(name, first_year)`插入一行，必须使用两个`?`；
- `insert_language_params`：参数元组，依次是`"Python"`和`1991`。

检测器会在独立数据库中执行它们。

In [ ]:
insert_language_sql = ""
insert_language_params = ()

In [ ]:
_check_connection = None
try:
    if "insert_language_sql" not in globals() or "insert_language_params" not in globals():
        print("❌ 请先运行上方答题单元格。")
    elif not isinstance(insert_language_sql, str):
        print("❌ insert_language_sql必须是字符串。")
    elif not isinstance(insert_language_params, tuple):
        print("❌ insert_language_params必须是元组。")
    elif insert_language_sql.count("?") != 2:
        print("❌ SQL必须恰好使用两个?占位符。")
    elif insert_language_params != ("Python", 1991):
        print("❌ 参数元组的值或顺序不正确。")
    else:
        _check_connection = sqlite3.connect(":memory:", autocommit=False)
        _check_connection.execute("CREATE TABLE languages(name TEXT, first_year INTEGER)")
        _check_connection.execute(insert_language_sql, insert_language_params)
        _check_row = _check_connection.execute("SELECT name, first_year FROM languages").fetchone()
        if _check_row == ("Python", 1991):
            print("✅ 小练习3通过：参数化INSERT正确。")
        else:
            print("❌ SQL执行了，但插入结果不正确：", _check_row)
except sqlite3.Error as _check_error:
    print("❌ SQL执行失败：", _check_error)
except Exception as _check_error:
    print("❌ 检测时发现问题：", _check_error)
finally:
    if _check_connection is not None:
        _check_connection.close()

<details><summary>完成后查看参考答案</summary>

SQL应使用`INSERT INTO languages(name, first_year) VALUES (?, ?)`，参数元组是`("Python", 1991)`。

</details>

## 10. `SELECT`、`fetchone()`和`fetchall()`

查询SQL的一般结构：

```sql
SELECT 列名 FROM 表名 WHERE 条件 ORDER BY 排序列 ASC
```

- `SELECT`指定要读取的列；显式列名比`SELECT *`更容易看懂接口契约；
- `WHERE`筛选符合条件的行；动态条件值仍然使用占位符；
- `ORDER BY year ASC`按年份升序；`DESC`表示降序；
- `cursor.fetchone()`没有参数，返回下一行；没有结果时返回`None`；
- `cursor.fetchall()`没有参数，返回剩余全部行组成的列表；没有结果时返回空列表`[]`。

`None`和空列表含义不同：查单个对象通常使用`None`表示不存在；查一组对象通常使用空列表表示集合为空。

In [ ]:
all_rows = demo_connection.execute(
    "SELECT id, title, year, rating, watched FROM movies ORDER BY year ASC"
).fetchall()
print("按年份排序：", [(row["title"], row["year"]) for row in all_rows])

one_row = demo_connection.execute(
    "SELECT id, title FROM movies WHERE id = ?",
    (2,),
).fetchone()
print("id=2：", dict(one_row))

missing_row = demo_connection.execute(
    "SELECT id, title FROM movies WHERE id = ?",
    (999,),
).fetchone()
print("不存在的id：", missing_row)

### 课堂小练习4：参数化筛选和排序

**目标：**使用`SELECT`、`WHERE`、占位符和`ORDER BY`。

从`demo_connection`的`movies`表中查询评分大于等于9.3的电影，只选择`title`和`year`，并按`year`升序。把SQL、参数元组和最终结果分别保存为：

- `select_movies_sql`
- `select_movies_params`
- `selected_movies`

正确结果的标题顺序应该是“机器人总动员”“星际穿越”。

In [ ]:
select_movies_sql = ""
select_movies_params = ()
selected_movies = []
# 使用demo_connection.execute(...).fetchall()得到selected_movies

In [ ]:
try:
    _check_required = ["select_movies_sql", "select_movies_params", "selected_movies"]
    _check_missing = [name for name in _check_required if name not in globals()]
    if _check_missing:
        print("❌ 请先运行答题格，缺少：", _check_missing)
    elif not isinstance(select_movies_sql, str) or "?" not in select_movies_sql:
        print("❌ select_movies_sql必须使用?占位符。")
    elif not isinstance(select_movies_params, tuple) or select_movies_params != (9.3,):
        print("❌ select_movies_params应是只含9.3的单元素元组。")
    elif not isinstance(selected_movies, list):
        print("❌ selected_movies必须是fetchall()返回的列表。")
    else:
        _check_titles = [row["title"] for row in selected_movies]
        _check_years = [row["year"] for row in selected_movies]
        if _check_titles != ["机器人总动员", "星际穿越"]:
            print("❌ 筛选结果或顺序不正确：", _check_titles)
        elif _check_years != [2008, 2014]:
            print("❌ year列的结果不正确：", _check_years)
        else:
            print("✅ 小练习4通过：筛选、参数和排序均正确。")
except Exception as _check_error:
    print("❌ 检测时发现问题：", _check_error)

<details><summary>完成后查看参考答案</summary>

SQL需要选择`title, year`，使用`WHERE rating >= ?`，最后使用`ORDER BY year ASC`；参数是`(9.3,)`，再对执行结果调用`fetchall()`。

</details>

## 11. `UPDATE`、`DELETE`和`rowcount`

修改已有行：

```sql
UPDATE movies SET watched = ? WHERE id = ?
```

删除已有行：

```sql
DELETE FROM movies WHERE id = ?
```

两者都必须谨慎使用`WHERE`。省略`WHERE`会修改或删除表中的所有行。

`execute()`返回的Cursor具有`rowcount`属性，表示这条`UPDATE`或`DELETE`影响了多少行：

- `rowcount == 1`：指定id存在，处理了一行；
- `rowcount == 0`：指定id不存在，没有行被处理。

`rowcount`是属性，不需要写括号。写操作之后仍然需要`commit()`。

In [ ]:
update_cursor = demo_connection.execute(
    "UPDATE movies SET watched = ? WHERE id = ?",
    (1, 2),
)
demo_connection.commit()
print("更新行数：", update_cursor.rowcount)

missing_delete = demo_connection.execute(
    "DELETE FROM movies WHERE id = ?",
    (999,),
)
demo_connection.commit()
print("删除不存在id时的行数：", missing_delete.rowcount)

### 课堂小练习5：准备参数化UPDATE

**目标：**更新一行并使用`rowcount`判断结果。

请设置：

- `update_movie_sql`：把指定电影的`watched`修改为指定值；
- `update_movie_params`：先传新值1，再传目标id 1。

必须使用两个`?`占位符。检测器会在独立数据库中执行。

In [ ]:
update_movie_sql = ""
update_movie_params = ()

In [ ]:
_check_connection = None
try:
    if "update_movie_sql" not in globals() or "update_movie_params" not in globals():
        print("❌ 请先运行上方答题单元格。")
    elif not isinstance(update_movie_sql, str) or update_movie_sql.count("?") != 2:
        print("❌ UPDATE语句必须恰好使用两个?占位符。")
    elif update_movie_params != (1, 1):
        print("❌ 参数顺序应先是新watched值1，再是目标id 1。")
    else:
        _check_connection = sqlite3.connect(":memory:", autocommit=False)
        _check_connection.execute("CREATE TABLE movies(id INTEGER PRIMARY KEY, watched INTEGER)")
        _check_connection.execute("INSERT INTO movies(id, watched) VALUES (?, ?)", (1, 0))
        _check_cursor = _check_connection.execute(update_movie_sql, update_movie_params)
        _check_value = _check_connection.execute("SELECT watched FROM movies WHERE id = ?", (1,)).fetchone()[0]
        if _check_cursor.rowcount != 1:
            print("❌ UPDATE没有恰好影响一行，rowcount是：", _check_cursor.rowcount)
        elif _check_value != 1:
            print("❌ id=1的watched没有被更新为1。")
        else:
            print("✅ 小练习5通过：UPDATE和参数顺序正确。")
except sqlite3.Error as _check_error:
    print("❌ SQL执行失败：", _check_error)
except Exception as _check_error:
    print("❌ 检测时发现问题：", _check_error)
finally:
    if _check_connection is not None:
        _check_connection.close()

<details><summary>完成后查看参考答案</summary>

SQL结构是`UPDATE movies SET watched = ? WHERE id = ?`，参数元组是`(1, 1)`。第一个1是新字段值，第二个1是目标主键。

</details>

## 12. 事务：`commit()`、`rollback()`和`close()`

**事务（transaction）**是一组应当作为整体处理的数据库操作。可以把它想成填写并最终确认一张银行转账单：

- `commit()`：确认当前事务，修改正式保存；
- `rollback()`：撤销当前尚未提交的修改；没有参数，返回`None`；
- `close()`：关闭Python与数据库的连接，释放文件和资源；没有参数，返回`None`，且不能代替`commit()`；
- 写操作失败时通常先`rollback()`，再把异常继续抛出；
- 无论成功或失败，最终都应该`close()`。

常见结构：

```python
connection = sqlite3.connect(path, autocommit=False)
try:
    connection.execute(...)
    connection.commit()
except sqlite3.Error:
    connection.rollback()
    raise
finally:
    connection.close()
```

`sqlite3.Error`是`sqlite3`模块提供的数据库异常基类；这里的`except sqlite3.Error`会捕获常见SQLite操作错误。`raise`不带参数时会继续抛出当前捕获的原异常，让调用者和测试知道真实原因。

### 文件持久化的可运行证明

这里首次使用：

- `TemporaryDirectory`来自标准库`tempfile`，进入`with`时创建临时目录，离开时自动删除；
- `Path`来自标准库`pathlib`，表示路径；`Path(directory) / "demo.sqlite3"`拼接文件路径；
- 这只是避免在课程目录留下演示数据库，真实项目会使用固定路径。

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as directory:
    persistence_path = Path(directory) / "persistence.sqlite3"

    write_connection = sqlite3.connect(persistence_path, autocommit=False)
    write_connection.execute("CREATE TABLE messages(content TEXT NOT NULL)")
    write_connection.execute("INSERT INTO messages(content) VALUES (?)", ("程序重启后仍存在",))
    write_connection.commit()
    write_connection.close()

    read_connection = sqlite3.connect(persistence_path, autocommit=False)
    persisted_row = read_connection.execute("SELECT content FROM messages").fetchone()
    read_connection.close()

    print("重新连接后读到：", persisted_row)

### 回滚示例：一组操作中途失败

下面先插入一条合法记录，再插入不符合`CHECK(year >= 1888)`的记录。`sqlite3.IntegrityError`是`sqlite3`模块提供的异常类，表示数据违反了数据库完整性约束。第二条触发它后执行`rollback()`，因此同一事务中第一条尚未提交的记录也会被撤销。

In [ ]:
before_count = demo_connection.execute("SELECT COUNT(*) AS count FROM movies").fetchone()["count"]

try:
    demo_connection.execute(
        "INSERT INTO movies(title, year) VALUES (?, ?)",
        ("事务中的合法电影", 2025),
    )
    demo_connection.execute(
        "INSERT INTO movies(title, year) VALUES (?, ?)",
        ("年份错误", 1800),
    )
    demo_connection.commit()
except sqlite3.IntegrityError as error:
    demo_connection.rollback()
    print("写入失败，已经回滚：", error)

after_count = demo_connection.execute("SELECT COUNT(*) AS count FROM movies").fetchone()["count"]
print("回滚前后行数相同：", before_count == after_count)

### 课堂小练习6：区分事务的共同步骤和两个分支

**目标：**区分提交、回滚和关闭的时机。

一次写操作不会先`commit`再`rollback`。请用字典`transaction_plan`明确表示共同步骤和互斥分支：

```python
transaction_plan = {
    "共同开始": [这里填写两个字符串],
    "成功分支": 这里填写一个字符串,
    "失败分支": [这里填写两个字符串],
    "共同结束": 这里填写一个字符串,
}
```

可用字符串只有：`"打开连接"`、`"执行写入"`、`"commit"`、`"rollback"`、`"重新抛出异常"`、`"finally中close"`。每个字符串恰好使用一次。

In [ ]:
transaction_plan = {
    "共同开始": [],
    "成功分支": "",
    "失败分支": [],
    "共同结束": "",
}

In [ ]:
try:
    _check_expected = {
        "共同开始": ["打开连接", "执行写入"],
        "成功分支": "commit",
        "失败分支": ["rollback", "重新抛出异常"],
        "共同结束": "finally中close",
    }
    if "transaction_plan" not in globals():
        print("❌ 请先运行上方答题单元格。")
    elif not isinstance(transaction_plan, dict):
        print("❌ transaction_plan必须是字典。")
    elif set(transaction_plan) != set(_check_expected):
        print("❌ 字典必须恰好包含共同开始、成功分支、失败分支、共同结束四个键。")
    elif transaction_plan != _check_expected:
        print("❌ 分支内容还不正确：注意commit和rollback不能放在同一条执行路线中。")
    else:
        print("✅ 小练习6通过：已经正确区分共同步骤与成功、失败分支。")
except Exception as _check_error:
    print("❌ 检测时发现问题：", _check_error)

<details><summary>完成后查看参考答案与说明</summary>

参考结构：`共同开始`是`["打开连接", "执行写入"]`；`成功分支`是`"commit"`；`失败分支`是`["rollback", "重新抛出异常"]`；`共同结束`是`"finally中close"`。这样能直接看出一次执行只会选择成功或失败分支之一。

</details>

## 13. 使用`sqlite3.Row`按列名读取

默认查询行是元组：

```python
(1, "星际穿越", 2014)
```

只看`row[1]`很难知道它是什么。连接后设置：

```python
connection.row_factory = sqlite3.Row
```

- `sqlite3.Row`是sqlite3提供的行类型；
- `row_factory`是Connection属性，影响之后创建的查询游标；
- 结果可用`row["title"]`按列名读取，也能用`row[0]`按位置读取；
- `dict(row)`返回普通Python字典，适合作为services层或API响应的输入。

必须在执行查询之前设置`row_factory`；修改它不会倒过来改变已经取得的旧行。

In [ ]:
row_result = demo_connection.execute(
    "SELECT id, title, watched FROM movies WHERE id = ?",
    (1,),
).fetchone()
print("按位置：", row_result[1])
print("按列名：", row_result["title"])
print("转字典：", dict(row_result))

### 课堂小练习7：把Row转换成业务字典

**目标：**使用列名读取并转换布尔含义。

已有`row_result`，请创建字典`movie_dict`，恰好包含：

- `id`：数据库id；
- `title`：标题；
- `watched`：把数据库中的0或1转换成Python布尔值。

In [ ]:
movie_dict = {}

In [ ]:
try:
    if "movie_dict" not in globals():
        print("❌ 请先运行上方答题单元格。")
    elif not isinstance(movie_dict, dict):
        print("❌ movie_dict必须是字典。")
    elif set(movie_dict) != {"id", "title", "watched"}:
        print("❌ movie_dict必须恰好包含id、title、watched三个键。")
    elif movie_dict.get("id") != 1 or movie_dict.get("title") != "星际穿越":
        print("❌ id或title没有从row_result正确取得。")
    elif movie_dict.get("watched") is not False:
        print("❌ watched应是Python的False，而不是整数0或其他值。")
    else:
        print("✅ 小练习7通过：Row已经转换成清晰的业务字典。")
except Exception as _check_error:
    print("❌ 检测时发现问题：", _check_error)

<details><summary>完成后查看参考答案思路</summary>

使用`row_result["id"]`和`row_result["title"]`按列名取值；对`row_result["watched"]`调用`bool(...)`，把0转换为`False`。

</details>

## 14. 数据访问层如何接到上一课的工程结构

上一课的调用链是：

```text
routers → services → 内存列表
```

加入数据库后可以变成：

```text
routers → services → repository → SQLite数据库文件
```

**repository（数据访问层）**是负责SQL和数据库连接的Python模块。它解决“怎样保存和查询”，services仍负责“业务规则”，routers仍负责HTTP。

| 层 | 负责 | 不应主要负责 |
|---|---|---|
| routers | HTTP路径、状态码、请求响应 | 编写SQL |
| services | 业务判断与流程 | 管理数据库连接细节 |
| repository | SQL、事务、行转换 | 决定HTTP状态码 |

本课作业只实现repository。下一课AI聊天后端项目会把它接进FastAPI。

## 15. 常见错误与定位顺序

| 现象 | 常见原因 | 先检查什么 |
|---|---|---|
| `no such table` | 没有初始化表，或连错数据库文件 | 数据库路径和`initialize`调用 |
| `database is locked` | 连接长期未关闭或同时写入 | 是否每个连接最终`close()` |
| 重启后数据消失 | 使用了`":memory:"`或忘记`commit()` | 连接参数和事务 |
| `Incorrect number of bindings` | `?`数量和参数数量不同 | SQL占位符及参数元组 |
| 字符串输入导致SQL错误 | 使用f字符串拼接SQL | 改用`?`占位符 |
| `fetchone()`后下标报错 | 查询没有结果，返回了`None` | 先判断`row is None` |
| 返回值全是0和1 | 数据库布尔列尚未转换 | 对字段使用`bool(...)` |

定位时先区分：连接失败、表结构问题、SQL语法问题、约束失败、事务未提交、结果转换错误。不要看到“数据库错误”就同时修改所有层。

## 16. 本课完整执行流程

```text
1. sqlite3.connect(...)打开连接
   ↓
2. 设置row_factory（需要按列名读取时）
   ↓
3. execute(SQL, 参数)把语句交给SQLite
   ↓
4a. 写操作：检查约束 → commit或rollback
4b. 读操作：fetchone或fetchall
   ↓
5. Row转换为字典或其他业务数据
   ↓
6. finally中close连接
```

Python负责调用方法、组织参数和转换结果；SQLite负责解析SQL、维护表、检查约束、执行事务和读写数据库文件。

In [ ]:
# 课堂演示结束，释放内存数据库连接。
demo_connection.close()
print("课堂数据库连接已关闭")

## 17. 课末作业：笔记数据库repository

### 任务背景

完成`note_repository.py`，让笔记不再保存在Python列表，而是写入SQLite文件。作业只考查数据访问层，不要求创建FastAPI路由。

### 允许修改范围

只修改`note_repository.py`中六个函数的`TODO`函数体：

- `initialize_database`
- `create_note`
- `list_notes`
- `get_note`
- `mark_note_completed`
- `delete_note`

顶部所需导入、`open_database()`和`_row_to_note()`已经提供，不需要修改。不要修改`test_note_repository.py`降低要求。

**建议实现顺序（不是答案）：**先完成`initialize_database`，再完成`create_note`，然后完成`list_notes`与`get_note`，最后完成更新和删除。后面的测试依赖数据库已经能初始化，因此每完成一组就重新运行测试。

### notes表结构契约

表名必须是`notes`，重复初始化不得删除原数据。

| 列 | SQLite类型 | 约束和默认值 | 数据由谁提供 |
|---|---|---|---|
| `id` | `INTEGER` | `PRIMARY KEY` | SQLite生成，调用者不提交 |
| `content` | `TEXT` | `NOT NULL`，长度1至200 | 调用者必须提供 |
| `category` | `TEXT` | `NOT NULL` | 调用者提供，Python参数默认`general` |
| `completed` | `INTEGER` | `NOT NULL DEFAULT 0`，只能是0或1 | 数据库创建时默认0，更新函数改成1 |

返回字典必须恰好包含`id`、`content`、`category`、`completed`，其中`completed`必须是Python的`bool`，不能是整数。

### 函数输入输出契约

| 函数 | 输入 | 返回值 | 必须支持的行为 |
|---|---|---|---|
| `initialize_database` | 数据库路径字符串 | `None` | 创建文件和表；重复调用保留数据 |
| `create_note` | 路径、content、可选category | 新生成的`int` id | 参数化插入并提交 |
| `list_notes` | 路径 | `list[dict]` | 按id升序；没有数据返回`[]` |
| `get_note` | 路径、整数id | `dict`或`None` | 参数化查询单行 |
| `mark_note_completed` | 路径、整数id | `bool` | 存在时更新为1并返回True；不存在返回False |
| `delete_note` | 路径、整数id | `bool` | 存在时删除并返回True；不存在返回False |

额外要求：

- 所有动态数据都必须使用`?`占位符，不能用f字符串拼SQL；
- 写操作成功后`commit()`；数据库错误时`rollback()`并继续抛出；
- 所有函数无论成功或失败都必须关闭连接；
- 空content或超过200字符由数据库约束拒绝，并保留`sqlite3.IntegrityError`；
- 类似`"'); DROP TABLE notes; --"`的content必须作为普通文本安全保存。

### 运行作业测试

在本课目录执行：

```powershell
python -m pytest test_note_repository.py -v
```

应收集**16个测试项**，全部完成后显示16 passed。

测试使用pytest的`tmp_path`夹具，为每个测试提供独立临时目录；测试结束后pytest清理它。测试会直接重新打开数据库文件，确认数据不是伪装保存在全局列表中。

## 18. 理解题

1. 为什么上一课的Python列表不属于持久化？SQLite数据库文件有什么不同？
2. 数据库、表、行、列和主键分别是什么？
3. `sqlite3.connect(...)`返回什么对象？`execute(...)`又返回什么对象？两者职责有什么区别？
4. 为什么不能用f字符串把用户输入拼进SQL？`?`占位符和第二个参数元组分别负责什么？
5. `commit()`、`rollback()`和`close()`分别做什么？为什么`close()`不能代替`commit()`？
6. `fetchone()`查不到数据与`fetchall()`查不到数据时，返回值分别是什么？为什么业务含义不同？
7. `sqlite3.Row`和`row_factory`解决了什么问题？repository在上一课的分层结构中位于哪里？

### 在这里填写理解题答案

1. 

2. 

3. 

4. 

5. 

6. 

7. 

## 19. 本课小结

你已经把上一课的“内存列表”向下替换为真正的数据存储：

- SQLite负责表、约束、SQL执行、事务和文件持久化；
- Python的`sqlite3`负责连接、绑定参数和取得结果；
- repository把SQL细节集中在数据访问层；
- services以后只需要调用repository，不必自己管理每条SQL；
- routers仍然只处理HTTP。

下一课“AI聊天后端项目”会把FastAPI工程结构、Pydantic模型和本课SQLite数据访问层组合成一个完整后端。

## 可选参考资料

- [Python官方文档：sqlite3](https://docs.python.org/3/library/sqlite3.html)
- [SQLite官方文档：CREATE TABLE](https://www.sqlite.org/lang_createtable.html)
- [SQLite官方文档：Transaction](https://www.sqlite.org/lang_transaction.html)

官方文档信息完整但默认读者已经了解游标和事务；第一次学习建议先按本Notebook顺序运行。